# 00 · Setup & Shared Ingredients

**Train vs. Tune vs. RAG — a side-by-side workshop**

This series shows three different ways to make an AI model useful for *your* topic, all on the
**same base model, the same knowledge, and the same test questions** so you can compare them fairly:

| Notebook | Approach | What changes | Who runs it |
|----------|----------|--------------|-------------|
| 01 | **Train from scratch** | The model learns everything from your data, from zero | Instructor (GPU) |
| 02 | **Fine-tune (LoRA)** | Nudges a pretrained model toward your task/format | Instructor (GPU) |
| 03 | **RAG** | *Nothing* in the model — it retrieves facts at question time | Anyone (CPU ok) |
| 04 | **Compare** | Same questions to all of them, side by side | Students (CPU ok) |

> **The big idea:** *need new **facts** → RAG; need new **behavior/format** → fine-tune; need a brand-new
> **capability** → train (and you'll see why that's expensive).*

This notebook defines the **shared ingredients** every other notebook uses: the base model, a small
knowledge corpus, and the fixed set of test questions.

## Install dependencies
The instructor's build notebooks (01, 02) need the full set; the student compare (04) and RAG (03)
need only the lighter ones. Installing everything here is fine.

In [ ]:
import sys, subprocess
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("transformers", "datasets", "accelerate", "peft", "trl",
    "sentence-transformers", "faiss-cpu", "huggingface_hub")
print("dependencies installed")

## Configuration
One base model for everything. We use a small model so training and tuning are quick and the whole
thing runs on modest hardware.

In [ ]:
from pathlib import Path

# Small instruct model used for fine-tuning (02) and RAG (03).
BASE_MODEL = "Qwen/Qwen3-1.7B"

# Where artifacts (trained model, LoRA adapter, RAG index) are saved/loaded.
ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)

# OPTIONAL: to share with students via Hugging Face, set your repo (e.g. "your-org/ttr-demo").
HF_REPO = None

print("Base model:", BASE_MODEL)
print("Artifacts dir:", ARTIFACTS.resolve())

## The shared knowledge corpus
A small, self-contained cybersecurity corpus — no downloads, so it always works. A few entries are
**deliberately fictional/proprietary** (a made-up university's policies). No pretrained model could
possibly know those — so later, **only RAG will answer them correctly.** That makes the difference
between the approaches unmistakable.

In [ ]:
import json

CORPUS = [
    {"title": "Phishing", "text": "Phishing is a social-engineering attack where an attacker sends fraudulent messages designed to trick a person into revealing sensitive information or installing malware."},
    {"title": "Ransomware", "text": "Ransomware is malware that encrypts a victim's files and demands payment, usually in cryptocurrency, in exchange for the decryption key."},
    {"title": "Zero-day", "text": "A zero-day is a software vulnerability unknown to the vendor, for which no patch yet exists, leaving systems exposed until one is released."},
    {"title": "Multi-factor authentication", "text": "Multi-factor authentication (MFA) requires two or more independent factors to verify identity, such as a password plus a one-time code from a phone."},
    {"title": "Least privilege", "text": "The principle of least privilege means giving each user or process only the access strictly required to do its job, and no more."},
    {"title": "SIEM", "text": "A Security Information and Event Management (SIEM) system collects and correlates logs from across an organization to detect and investigate security incidents."},
    {"title": "Defense in depth", "text": "Defense in depth is a strategy of layering multiple, independent security controls so that if one fails, others still protect the system."},
    {"title": "SQL injection", "text": "SQL injection is an attack that inserts malicious SQL into an application's query, letting an attacker read or modify a database they should not access."},
    {"title": "Patch management", "text": "Patch management is the process of regularly identifying, testing, and applying software updates to fix security vulnerabilities."},
    {"title": "Incident response", "text": "Incident response is the organized approach to detecting, containing, eradicating, and recovering from a cybersecurity incident, followed by lessons learned."},
    # --- Fictional / proprietary facts: only RAG can know these ---
    {"title": "Redlake University password policy", "text": "Redlake University requires all account passwords to be at least 16 characters long and rotated every 180 days. Reuse of the last 10 passwords is prohibited."},
    {"title": "Redlake University SOC hours", "text": "The Redlake University Security Operations Center (SOC) is staffed 24/7, and all suspected incidents must be reported to soc@redlake.example within 30 minutes of discovery."},
    {"title": "Redlake VPN requirement", "text": "Remote access to Redlake University systems requires the GlobalGuard VPN client and a hardware security key; software one-time codes are not accepted for VPN login."},
]

with open(ARTIFACTS / "corpus.json", "w", encoding="utf-8") as f:
    json.dump(CORPUS, f, indent=2)
print(f"Saved {len(CORPUS)} corpus entries to {ARTIFACTS/'corpus.json'}")

## The fixed test questions
The same questions go to every approach in Notebook 04. They're chosen so each one *exposes* a
difference — note the **"who should win"** column.

In [ ]:
TEST_PROMPTS = [
    {"type": "corpus fact",     "q": "What is Redlake University's password policy?",            "win": "RAG (it's a private fact no model was trained on)"},
    {"type": "corpus fact",     "q": "How quickly must incidents be reported at Redlake, and to whom?", "win": "RAG"},
    {"type": "general fact",    "q": "What is phishing?",                                        "win": "Base/tune/RAG all OK (common knowledge)"},
    {"type": "format/behavior", "q": "Define 'least privilege' in one sentence for a beginner.",  "win": "Fine-tuned (learned the style)"},
    {"type": "reasoning",       "q": "A new employee reuses one short password everywhere. Which two ideas from security best practice would help, and why?", "win": "Larger/instruct models; shows reasoning"},
    {"type": "out-of-scope",    "q": "Write a haiku about the ocean.",                            "win": "Base/instruct (shows specialization trade-offs)"},
]

with open(ARTIFACTS / "test_prompts.json", "w", encoding="utf-8") as f:
    json.dump(TEST_PROMPTS, f, indent=2)
print(f"Saved {len(TEST_PROMPTS)} test prompts.")
for p in TEST_PROMPTS:
    print(f"  [{p['type']:14}] {p['q']}")

✅ **Setup done.** The corpus and test prompts are saved in `artifacts/`. Next:
- **Instructor:** run **01 (train)** and **02 (fine-tune)** on a GPU, then **03 (RAG)**.
- **Students:** once the instructor shares the `artifacts/` folder, jump to **04 (compare)**.